<a href="https://colab.research.google.com/github/manubastidas/programacionCientifica/blob/eval2-MarulandaDur%C3%A1n/Evaluacion2/MarulandaDur%C3%A1n/solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
##Inicialización de LaTeX

!sudo apt-get update
!sudo apt-get install texlive-latex-extra texlive-fonts-recommended dvipng cm-super
!pip install -q jax jaxlib optax

import numpy as np

from scipy.special import comb
from scipy.interpolate import CubicSpline, make_interp_spline

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "font.size": 14,

    # Ejes y Ticks
    "axes.labelsize": 16,
    "axes.titlesize": 18,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.direction": "in",
    "ytick.direction": "in",

    # Grid
    "grid.color"    : "gray",
    "grid.linewidth": 0.3,
    "grid.alpha"    : 0.3,
    "grid.linestyle": "--",

    # Estética
    "figure.dpi"        : 120,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "savefig.bbox"      : "tight",
    "savefig.dpi"       : 300,
})

print('Configuración OK')

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
cm-super is already the newest version (0.3.4-17).
dvipng is already the newest version (1.15-

In [19]:
import numpy as np
import jax.numpy as jnp
import jax
from jax import random
import matplotlib.pyplot as plt

# Carga de datos
datos = np.load('datos_6098.npz')
X = datos['X'] # Forma: (30, 256)
y = datos['y'] # Forma: (30,) con etiquetas de régimen 0, 1, 2
t = datos['t'] # Forma: (256,)

# Extraemos una señal por régimen para visualizaciones futuras
X_reg0 = X[y == 0]
X_reg1 = X[y == 1]
X_reg2 = X[y == 2]

In [20]:
def construir_capa_fourier(t, K=25):
    """Construye la matriz W1 con filas de senos y cosenos."""
    # La matriz debe transformar de 256 a 50 (2K)
    W1 = np.zeros((2 * K, len(t)))

    # Asumimos que t va de 0 a un periodo T. Si t está normalizado entre 0 y 1:
    T = t[-1] - t[0] + (t[1] - t[0])

    for k in range(1, K + 1):
        # Frecuencias k
        W1[2*(k-1), :] = np.cos(2 * np.pi * k * t / T)
        W1[2*(k-1)+1, :] = np.sin(2 * np.pi * k * t / T)

    return jnp.array(W1)

W1_fourier = construir_capa_fourier(t, K=25)

# Evidencia de comparación con np.fft
x_prueba = X[0]
w1_out = W1_fourier @ x_prueba
fft_out = np.fft.fft(x_prueba)

# Reporte de la energía media del dataset
energia_media = (X**2).mean()
print(f"Energía media del dataset: {energia_media:.3f}")

Energía media del dataset: 0.786


Las entradas de np.fft son señales temporales, y las salidas son coeficientes complejos en el dominio de la frecuencia.

En este caso se hace con una sola señal para verificar la inicialización de $W_1$.

Reportamos la energía media establecer un umbral base que usaremos en la sección de compresión